# Generate from one completed SOURCE run

Run Notebook 01 as SOURCE first. Keep `notebooks/` and `src/` together. The cells below collect parameters, load pinned aggregate profiles, generate records, display audits, and save/validate the Delta outputs.

The reusable code retains the existing sampling order and legacy fallbacks. Model text is never executed. Only reviewed proposals with valid provenance can select predefined rule implementations.

In [ ]:
import sys
from pathlib import Path

# Keep notebooks/ and src/ together; no package installation is needed.
repo_dirs = (Path.cwd(), Path.cwd().parent, *(Path(path) for path in sys.path if path))
for repo_dir in repo_dirs:
    src_dir = repo_dir if repo_dir.name == "src" else repo_dir / "src"
    if (src_dir / "synthetic_banking" / "config.py").is_file():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Keep notebooks/ and src/ together, or add the project's src/ folder to the Python path.")

## 1. Parameters

Leave `source_run_id` blank to select the latest completed SOURCE run once. For a controlled pilot, supply the exact SOURCE run, batch, and one to three reviewed proposal IDs. Customer count and seed use the shared defaults below.

In [ ]:
from synthetic_banking.config import GenerationSettings, DEFAULT_CUSTOMERS, DEFAULT_SEED

dbutils.widgets.text("source_run_id", "", "Completed SOURCE run ID")
dbutils.widgets.text("pilot_batch_id", "", "Pilot proposal batch ID")
dbutils.widgets.text("pilot_proposal_ids", "", "One to three reviewed proposal IDs, comma-separated")
dbutils.widgets.dropdown("workflow_mode", "OFF", ["OFF", "ON"])
dbutils.widgets.text("proposal_batch_id", "")
dbutils.widgets.text("approval_fingerprint", "")
workflow_mode = dbutils.widgets.get("workflow_mode") == "ON"
requested_source_run_id = dbutils.widgets.get("source_run_id").strip()
settings = GenerationSettings(
    num_customers=DEFAULT_CUSTOMERS,  # Set the population size for this run here.
    seed=DEFAULT_SEED,
    proposal_batch_id=dbutils.widgets.get("proposal_batch_id").strip() if workflow_mode else "",
    pilot_batch_id=dbutils.widgets.get("pilot_batch_id").strip(),
    pilot_proposal_ids=tuple(value.strip() for value in dbutils.widgets.get("pilot_proposal_ids").split(",") if value.strip()),
)
if workflow_mode:
    from synthetic_banking.workflow import check_approvals, enforce_approval, required_id
    required_id(requested_source_run_id, "SOURCE run_id")
    required_id(settings.proposal_batch_id, "proposal batch_id")
    enforce_approval(check_approvals(spark, requested_source_run_id, settings.proposal_batch_id),
                     dbutils.widgets.get("approval_fingerprint").strip())

## 2. Load pinned SOURCE profiles

Every profile and proposal used below belongs to the selected completed SOURCE run. Convenience views and `current_run` are not generation inputs.

In [ ]:
from synthetic_banking.profiles import load_generation_profiles

profiles = load_generation_profiles(
    spark, requested_source_run_id,
    pilot_batch_id=settings.pilot_batch_id,
    pilot_proposal_ids=settings.pilot_proposal_ids,
)
source_run_id = profiles.source_run_id
print(f"Pinned SOURCE: {source_run_id}; seed: {settings.seed}")

## 3. Generate and inspect

`generate_dataset` calls the existing samplers, temporal constraints, controlled rule registry, and customer-relationship generator. No tables are written until the next stage.

In [ ]:
from synthetic_banking.generation import generate_dataset

if workflow_mode:
    enforce_approval(check_approvals(spark, source_run_id, settings.proposal_batch_id),
                     dbutils.widgets.get("approval_fingerprint").strip())

result = generate_dataset(spark, profiles, settings)

In [ ]:
from dataclasses import asdict
import pandas as pd

display(pd.DataFrame([{"table": name, "row_count": len(frame)} for name, frame in result.tables.items()]))
display(pd.DataFrame([asdict(event) for event in result.rule_events]))
display(pd.DataFrame(result.date_diagnostics))
print("Customer relationship plan:", result.relationship_plan)

## 4. Save and validate

This overwrites the six output tables in the configured generated schema, appends both audits, and runs the existing validation gates. A generation is published as COMPLETED only after validation succeeds. Record the generation ID for Notebook 01 in GENERATED mode.

In [ ]:
from synthetic_banking.storage import save_generation

if workflow_mode:
    enforce_approval(check_approvals(spark, source_run_id, settings.proposal_batch_id),
                     dbutils.widgets.get("approval_fingerprint").strip())

generation_id = save_generation(spark, result)
print(f"Use generation_id={generation_id} in Notebook 01 with profile_type=GENERATED.")
if workflow_mode:
    from synthetic_banking.workflow import require_generation, publish_values
    require_generation(spark, source_run_id, settings.proposal_batch_id, generation_id)
    publish_values(dbutils, source_run_id=source_run_id, batch_id=settings.proposal_batch_id,
                   generation_id=generation_id)

## Row-level dataset packaging is gated

The legacy ZIP utility is disabled because arbitrary existing CSVs cannot be bound to a privacy-assessed generation. Aggregate reporting remains available. A future generation-bound dataset exporter must enforce `privacy.require_row_level_release` before any download. Privacy screening PASS alone is not anonymization certification or distribution approval.


In [ ]:
# Row-level packaging is intentionally disabled; no dataset archive is created here.
# See docs/privacy.md for the mandatory gate and limitations.
